# 02 - Visual embedding (single video)

Nh?n output t? `01-shot-keyframe-single-video.ipynb`, encode OpenCLIP cho to?n b? keyframe v? l?u vector ?? import v?o Milvus.


In [ ]:
%pip install -q open_clip_torch pandas pyarrow pillow


In [ ]:
from __future__ import annotations

import json
import shutil
from pathlib import Path

import numpy as np
import open_clip
import pandas as pd
import torch
from PIL import Image

VIDEO_ID = 'L21_V001'
PIPELINE_VERSION = 'aicv3-visual'
MODEL_NAME = 'ViT-B-32'
MODEL_PRETRAINED = 'laion2b_s34b_b79k'
BATCH_SIZE = 64

matches = list(Path('/kaggle/input').rglob('keyframes.parquet'))
if not matches:
    raise FileNotFoundError('Kh?ng t?m th?y keyframes.parquet t? notebook 01')
# Some Kaggle inputs attach an entire batch dataset (many videos' 01-output
# folders at once). Picking matches[0] blindly would silently embed the wrong
# video under this notebook's VIDEO_ID. Prefer a path that names VIDEO_ID; if
# more than one match exists and none names it, fail loudly instead of guessing.
video_matches = [path for path in matches if VIDEO_ID.lower() in str(path).lower()]
if video_matches:
    KEYFRAMES_PATH = video_matches[0]
elif len(matches) == 1:
    KEYFRAMES_PATH = matches[0]
else:
    raise FileNotFoundError(
        f'{len(matches)} tep keyframes.parquet nhung khong tep nao khop '
        f'{VIDEO_ID}; kiem tra lai Kaggle input de tranh embed nham video khac.'
    )
INPUT_ROOT = KEYFRAMES_PATH.parent
if not (INPUT_ROOT / 'frames').exists():
    # M?t s? Kaggle Dataset ??t parquet trong th? m?c con. T?m th? m?c ch?a frame.
    frame_roots = list(INPUT_ROOT.rglob('frames'))
    if not frame_roots:
        raise FileNotFoundError('Kh?ng t?m th?y th? m?c frames c?nh keyframes.parquet')
    FRAME_ROOT = frame_roots[0]
else:
    FRAME_ROOT = INPUT_ROOT / 'frames'

OUTPUT_ROOT = Path('/kaggle/working/02-visual-embedding-output')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    raise RuntimeError('H?y b?t GPU trong Kaggle Notebook Settings')

keyframes_df = pd.read_parquet(KEYFRAMES_PATH).sort_values('ordinal').reset_index(drop=True)
# Defence in depth: even a correctly-chosen keyframes.parquet could concatenate
# rows for multiple videos. Never embed anything outside VIDEO_ID.
keyframes_df = keyframes_df[keyframes_df['video_id'] == VIDEO_ID].reset_index(drop=True)
if keyframes_df.empty:
    raise ValueError(f'{KEYFRAMES_PATH} khong chua keyframe nao cua {VIDEO_ID}')
print('Input:', INPUT_ROOT)
print('Keyframes:', len(keyframes_df))
print('Device:', torch.cuda.get_device_name(0))


In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=MODEL_PRETRAINED, device=DEVICE
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.eval()

vectors = []
missing = []
with torch.inference_mode():
    for start in range(0, len(keyframes_df), BATCH_SIZE):
        batch_rows = keyframes_df.iloc[start:start + BATCH_SIZE]
        images = []
        valid_indices = []
        for index, row in batch_rows.iterrows():
            frame_path = INPUT_ROOT / row.frame_path
            if not frame_path.exists():
                frame_path = FRAME_ROOT.parent / row.frame_path
            if not frame_path.exists():
                missing.append({'keyframe_id': row.keyframe_id, 'frame_path': str(frame_path)})
                continue
            try:
                images.append(preprocess(Image.open(frame_path).convert('RGB')))
                valid_indices.append(index)
            except Exception as exc:
                missing.append({'keyframe_id': row.keyframe_id, 'frame_path': str(frame_path), 'error': str(exc)})
        if images:
            batch = torch.stack(images).to(DEVICE)
            encoded = model.encode_image(batch)
            encoded = torch.nn.functional.normalize(encoded.float(), dim=-1)
            vectors.extend(encoded.cpu().numpy().astype('float32'))
        if start == 0 or (start + len(batch_rows)) % (BATCH_SIZE * 4) == 0 or start + len(batch_rows) == len(keyframes_df):
            print(f'Embedding: {min(start + len(batch_rows), len(keyframes_df))}/{len(keyframes_df)}')

if missing:
    raise RuntimeError(f'Thi?u ho?c l?i {len(missing)} frame; v? d?: {missing[:3]}')
visual_vectors = np.asarray(vectors, dtype='float32')
if len(visual_vectors) != len(keyframes_df):
    raise RuntimeError(f'S? vector {len(visual_vectors)} kh?c s? keyframe {len(keyframes_df)}')

np.save(OUTPUT_ROOT / 'visual.npy', visual_vectors)
records_df = keyframes_df[['keyframe_id', 'video_id', 'shot_id', 'frame_idx', 'pts_time', 'frame_path']].copy()
records_df['embedding_row'] = np.arange(len(records_df), dtype='int64')
records_df['embedding_model'] = f'{MODEL_NAME}/{MODEL_PRETRAINED}'
records_df['embedding_dim'] = visual_vectors.shape[1]
records_df.to_parquet(OUTPUT_ROOT / 'embedding-records.parquet', index=False)

summary = {
    'stage': 'visual-embedding',
    'pipeline_version': PIPELINE_VERSION,
    'video_id': VIDEO_ID,
    'keyframes': int(len(keyframes_df)),
    'embedding_shape': list(visual_vectors.shape),
    'model': f'{MODEL_NAME}/{MODEL_PRETRAINED}',
    'metric': 'IP with L2-normalized vectors',
}
(OUTPUT_ROOT / 'visual-summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Ki?m tra v? ??ng g?i output visual embedding.
assert np.isfinite(visual_vectors).all()
norms = np.linalg.norm(visual_vectors, axis=1)
assert np.allclose(norms, 1.0, atol=1e-4)
assert records_df['embedding_row'].is_unique

archive = shutil.make_archive(
    '/kaggle/working/02-visual-embedding-L21_V001',
    'zip',
    root_dir=OUTPUT_ROOT,
)
print('Download:', archive)

# Gi?i ph?ng model sau khi ?? ghi visual.npy.
del model, tokenizer, preprocess, visual_vectors, vectors
import gc
gc.collect()
torch.cuda.empty_cache()
print('Released OpenCLIP memory')
